In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
df = pd.read_csv('sales_data.csv')
print(df.head())
print(df.info())

   Product_ID   Sale_Date Sales_Rep Region  Sales_Amount  Quantity_Sold  \
0        1052  2023-02-03       Bob  North       5053.97           18.0   
1        1093  2023-04-21       Bob   West           NaN           17.0   
2        1015  2023-09-21     David  South       4631.23           30.0   
3        1072  2023-08-24       Bob    NaN       2167.94           39.0   
4        1061  2023-03-24   Charlie    NaN       3750.20            NaN   

  Product_Category  Unit_Cost  Unit_Price Customer_Type  Discount  \
0        Furniture     152.75      267.22     Returning      0.09   
1              NaN    3816.39     4209.44     Returning      0.11   
2             Food        NaN      371.40     Returning      0.20   
3         Clothing        NaN     4467.75           New      0.02   
4      Electronics     637.37      692.71           New      0.08   

  Payment_Method Sales_Channel Region_and_Sales_Rep  
0           Cash        Online            North-Bob  
1           Cash        Re

In [3]:
df.isnull().sum()

Product_ID              0
Sale_Date               0
Sales_Rep               2
Region                  4
Sales_Amount            4
Quantity_Sold           3
Product_Category        5
Unit_Cost               3
Unit_Price              0
Customer_Type           0
Discount                0
Payment_Method          0
Sales_Channel           0
Region_and_Sales_Rep    0
dtype: int64

In [11]:
df = pd.read_csv('sales_data.csv')
df.dropna(subset=['Sales_Amount', 'Quantity_Sold'], inplace=True)
df['Sales_Rep'] = df['Sales_Rep'].fillna('Unknown Rep')
df['Region'] = df['Region'].fillna('Unknown Region')
most_frequent_cat = df['Product_Category'].mode()[0]
df['Product_Category'] = df['Product_Category'].fillna(most_frequent_cat)
df['Unit_Cost'] = df.groupby('Product_Category')['Unit_Cost'].transform(lambda x: x.fillna(x.median()))
df['Unit_Cost'] = df['Unit_Cost'].fillna(df['Unit_Cost'].median())
print("--- Missing Values Remaining ---")
print(df.isnull().sum())

--- Missing Values Remaining ---
Product_ID              0
Sale_Date               0
Sales_Rep               0
Region                  0
Sales_Amount            0
Quantity_Sold           0
Product_Category        0
Unit_Cost               0
Unit_Price              0
Customer_Type           0
Discount                0
Payment_Method          0
Sales_Channel           0
Region_and_Sales_Rep    0
dtype: int64


In [12]:
#  predict the 'Sales_Amount' column
X = df.drop(columns=['Sales_Amount']) 
y = df['Sales_Amount']                 

In [19]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

df = pd.read_csv('sales_data.csv')

df.dropna(subset=['Sales_Amount', 'Quantity_Sold'], inplace=True)
df['Sales_Rep'] = df['Sales_Rep'].fillna('Unknown Rep')
df['Region'] = df['Region'].fillna('Unknown Region')
df['Product_Category'] = df['Product_Category'].fillna(df['Product_Category'].mode()[0])
df['Unit_Cost'] = df.groupby('Product_Category')['Unit_Cost'].transform(lambda x: x.fillna(x.median()))

# 1. Extract features from the Date column
df['Sale_Date'] = pd.to_datetime(df['Sale_Date'])
df['Sale_Year'] = df['Sale_Date'].dt.year
df['Sale_Month'] = df['Sale_Date'].dt.month
df['Sale_Day'] = df['Sale_Date'].dt.day
df = df.drop(columns=['Sale_Date'])

# 2. CONVERT ALL TEXT COLUMNS INTO NUMBERS (One-Hot Encoding)
categorical_cols = [
    'Sales_Rep', 
    'Region', 
    'Product_Category', 
    'Payment_Method', 
    'Sales_Channel', 
    'Customer_Type'
]
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# 3. Drop unneeded raw identifier columns
cols_to_drop = ['Product_ID', 'Region_and_Sales_Rep']
df = df.drop(columns=cols_to_drop, errors='ignore') 

# 4. Split into X (Features) and y (Target)
X = df.drop(columns=['Sales_Amount'])  
y = df['Sales_Amount']

# 5. Split into Train/Test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [20]:
model = RandomForestRegressor(n_estimators=100, random_state=42)
print("Training the model... Please wait...")
model.fit(X_train, y_train)
print("Training complete without any errors!")

Training the model... Please wait...
Training complete without any errors!


In [21]:
# Pass your hidden test features to see what the model thinks sales should be
predictions = model.predict(X_test)

# Print the first 5 predicted sales numbers vs actual sales numbers
print("Predicted Sales vs Actual Sales:")
for i in range(5):
    print(f"Prediction: ${predictions[i]:.2f} | Actual: ${y_test.iloc[i]:.2f}")

Predicted Sales vs Actual Sales:
Prediction: $4884.31 | Actual: $8417.07
Prediction: $5671.68 | Actual: $6882.83
Prediction: $5535.91 | Actual: $1697.49
Prediction: $5791.65 | Actual: $7255.10
Prediction: $5318.97 | Actual: $8635.81
